In [10]:
import networkx as nx
import pandas as pd
from pathlib import Path

from src.utils.data_loader import ROOT

# Caminhos de entrada e saída
GRAPH_PATH = ROOT / "artifacts/graph/network_disparity.graphml"
OUTPUT_CSV = ROOT / "reports/graph_analysis/louvain_communities_mapping.csv"

def map_and_check_communities():
    print('='*50)
    print("1. Carregando Grafo e Mapeando Comunidades (Louvain)...")
    
    try:
        G = nx.read_graphml(GRAPH_PATH)
    except FileNotFoundError:
        print("Erro: Arquivo .graphml não encontrado. Rode o pipeline de extração primeiro.")
        return

    print(f"Grafo carregado com sucesso: {G.number_of_nodes()} nós e {G.number_of_edges()} arestas.")

    # --- NOVO PASSO: Extrair o Componente Gigante ---
    #print("\nFiltrando ilhas isoladas (mantendo apenas o Componente Gigante)...")
    #componentes = list(nx.connected_components(G))
    #maior_componente = max(componentes, key=len)
    #G = G.subgraph(maior_componente).copy()
    #print(f"O Componente Gigante possui {G.number_of_nodes()} nós e {G.number_of_edges()} arestas.")
    #print(f"Foram ignoradas {len(componentes) - 1} pequenas ilhas isoladas.")

    # 1. Aplicar o Algoritmo de Louvain com Resolução Ajustada
    print("\nExecutando clusterização de comunidades (Louvain)...")
    
    # AJUSTE AQUI: Resolução < 1.0 cria comunidades MAIORES.
    # Tente 0.8 para uma fusão leve, ou 0.5 para macro-comunidades bem grandes.
    RESOLUCAO = 0.5
    
    communities = nx.community.louvain_communities(G, weight='peso_jaccard', resolution=RESOLUCAO)

    # Calculamos a centralidade de grau para saber quais são os fóruns principais de cada grupo
    degrees = dict(G.degree())
    
    # Subreddits alvo para rastreamento qualitativo
    target_subs = [
        'askmrp', 'blackpillscience', 'marriedredpill', 
        'redpillwomen', 'seduction', 'semenretention', 'theredpill'
    ]
    
    # Estruturando os dados
    data = []
    for comm_id, comm in enumerate(communities):
        for sub in comm:
            data.append({
                'subreddit': sub,
                'community_id': comm_id,
                'degree_centrality': degrees[sub]
            })
            
    df_comm = pd.DataFrame(data)
    
    # Ordenando para que os fóruns mais conectados fiquem no topo de suas comunidades
    df_comm = df_comm.sort_values(by=['community_id', 'degree_centrality'], ascending=[True, False])
    
    print(f"Rede estrutural dividida com sucesso em {len(communities)} macro-comunidades.")
    
    print("\n" + "="*50)
    print("2. RASTREAMENTO DOS SUBREDDITS ALVO (MANOSFERA)")
    print("="*50)
    
    for target in target_subs:
        if target in G.nodes():
            comm_id = df_comm[df_comm['subreddit'] == target]['community_id'].values[0]
            print(f"[✓] {target.upper()} SOBREVIVEU! Pertence à Macro-Comunidade {comm_id}.")
        else:
            print(f"[X] {target.upper()} foi descartado pelo filtro de disparidade (ruído).")
            
    print("\n" + "="*50)
    print("3. VISÃO GERAL DAS MAIORES COMUNIDADES (TOP 5 LÍDERES)")
    print("="*50)
    
    # Selecionamos as 10 comunidades com o maior número de subreddits
    maiores_comunidades = df_comm['community_id'].value_counts().head(10).index
    
    for comm_id in maiores_comunidades:
        subs_da_comunidade = df_comm[df_comm['community_id'] == comm_id]
        tamanho = len(subs_da_comunidade)
        # Pegamos os 5 subreddits com maior grau (os hubs)
        top_5 = subs_da_comunidade['subreddit'].head(5).tolist()
        
        print(f"Macro-Comunidade {comm_id} (Total: {tamanho} fóruns)")
        print(f"Hubs Principais: {', '.join(top_5)}")
        print("-" * 40)


if __name__ == "__main__":
    map_and_check_communities()

1. Carregando Grafo e Mapeando Comunidades (Louvain)...
Grafo carregado com sucesso: 4873 nós e 11961 arestas.

Executando clusterização de comunidades (Louvain)...
Rede estrutural dividida com sucesso em 475 macro-comunidades.

2. RASTREAMENTO DOS SUBREDDITS ALVO (MANOSFERA)
[✓] ASKMRP SOBREVIVEU! Pertence à Macro-Comunidade 215.
[✓] BLACKPILLSCIENCE SOBREVIVEU! Pertence à Macro-Comunidade 37.
[✓] MARRIEDREDPILL SOBREVIVEU! Pertence à Macro-Comunidade 215.
[X] REDPILLWOMEN foi descartado pelo filtro de disparidade (ruído).
[✓] SEDUCTION SOBREVIVEU! Pertence à Macro-Comunidade 153.
[✓] SEMENRETENTION SOBREVIVEU! Pertence à Macro-Comunidade 343.
[✓] THEREDPILL SOBREVIVEU! Pertence à Macro-Comunidade 414.

3. VISÃO GERAL DAS MAIORES COMUNIDADES (TOP 5 LÍDERES)
Macro-Comunidade 45 (Total: 450 fóruns)
Hubs Principais: television, finalfantasy, netflix, spiderman, entertainment
----------------------------------------
Macro-Comunidade 46 (Total: 439 fóruns)
Hubs Principais: youseeingthisshi

In [14]:
import pandas as pd
import networkx as nx
from pathlib import Path

# Ajuste a importação conforme a estrutura do seu projeto
from src.utils.data_loader import ROOT, load_raw_data 

GRAPH_PATH = ROOT / "artifacts/graph/network_disparity.graphml"
PATH_TOXICITY = ROOT / "data" / "processed" / "toxicidade_perspective_COMPLETO.csv"
OUTPUT_CSV = ROOT / "reports/graph_analysis/ranking_comunidades_toxicas.csv"

def rank_toxic_communities():
    print('='*50)
    print("1. Carregando Grafo e Extraindo Comunidades...")
    try:
        G = nx.read_graphml(GRAPH_PATH)
    except FileNotFoundError:
        print("Erro: network_disparity.graphml não encontrado.")
        return
    """
    # --- NOVO PASSO: Extrair o Componente Gigante ---
    print("\nFiltrando ilhas isoladas (mantendo apenas o Componente Gigante)...")
    componentes = list(nx.connected_components(G))
    maior_componente = max(componentes, key=len)
    G = G.subgraph(maior_componente).copy()
    print(f"O Componente Gigante possui {G.number_of_nodes()} nós e {G.number_of_edges()} arestas.")
    print(f"Foram ignoradas {len(componentes) - 1} pequenas ilhas isoladas.")
    """
    # Executamos o Louvain na rede inteira (incluindo as ilhas)
    communities = nx.community.louvain_communities(G, weight='peso_jaccard', resolution=0.5, seed=42)
    
    # Dicionário mapeando {subreddit: ID_da_Comunidade}
    sub2comm = {}
    for comm_id, comm in enumerate(communities):
        for sub in comm:
            sub2comm[sub] = comm_id
            
    print(f"Total de {len(communities)} macro-comunidades mapeadas na topologia.")

    print("\n2. Carregando Corpus Dinamicamente (Apenas posts válidos para NLP)...")
    df_textos = load_raw_data(columns=['id', 'subreddit'], only_valid_ids=True)
    
    # Mantém apenas os posts de subreddits que sobreviveram ao filtro estatístico
    subreddits_validos = list(sub2comm.keys())
    df_corpus = df_textos[df_textos['subreddit'].isin(subreddits_validos)].copy()
    
    # Mapeia cada post para o ID da comunidade do seu subreddit
    df_corpus['community'] = df_corpus['subreddit'].map(sub2comm).astype(int)
    print(f"Posts retidos no backbone estrutural: {len(df_corpus)}")

    print("\n3. Carregando Dados do Perspective API e Cruzando...")
    # Lê os dados de toxicidade (Altere para read_csv se a sua base for .csv)
    df_tox = pd.read_csv(PATH_TOXICITY) 
    
    # Mescla o corpus com as pontuações de toxicidade usando o 'id'
    df_merged = df_corpus.merge(df_tox, on='id', how='inner')
    print(f"Posts cruzados com sucesso: {len(df_merged)}")

    print("\n4. Agregando Estatísticas de Toxicidade por Comunidade...")
    # Agrupa pelo ID da comunidade e calcula as médias
    comm_stats = df_merged.groupby('community').agg(
        num_posts=('id', 'count'),
        num_subs=('subreddit', 'nunique'),
        avg_toxicity=('perspective_toxicity', 'mean'),
        avg_severe=('severe_toxicity', 'mean'),
        avg_identity=('identity_attack', 'mean'),
        avg_insult=('insult', 'mean'),
        avg_threat=('threat', 'mean'),
        avg_profanity=('profanity', 'mean')
    ).reset_index()

    # Identificando o "Subreddit Hub" (o fórum com mais posts dentro da comunidade para dar contexto)
    top_subs = df_merged.groupby(['community', 'subreddit']).size().reset_index(name='count')
    top_subs = top_subs.sort_values(['community', 'count'], ascending=[True, False]).drop_duplicates('community')
    
    comm_stats = comm_stats.merge(top_subs[['community', 'subreddit']], on='community', how='left')
    comm_stats.rename(columns={'subreddit': 'hub_principal'}, inplace=True)

    # Filtro de Confiança: Ignorar ilhas minúsculas com menos de 50 posts válidos 
    # para não distorcer a média geral com amostras irrelevantes
    comm_stats_validas = comm_stats[comm_stats['num_posts'] >= 50].copy()

    # Ordena o ranking pela Toxicidade Geral (Perspective Toxicity)
    ranking = comm_stats_validas.sort_values('avg_toxicity', ascending=False)

    print("\n" + "="*50)
    print("TOP 10 COMUNIDADES MAIS TÓXICAS DO ECOSSISTEMA")
    print("="*50)
    
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    
    # Exibe as colunas essenciais no terminal
    top_10 = ranking.head(10)[['community', 'hub_principal', 'num_subs', 'num_posts', 'avg_toxicity', 'avg_identity']]
    print(top_10.to_string(index=False))
    
    # Salva o arquivo CSV completo para a redação da dissertação
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    ranking.to_csv(OUTPUT_CSV, index=False)
    print(f"\nRanking detalhado salvo em:\n{OUTPUT_CSV}")

if __name__ == "__main__":
    rank_toxic_communities()

1. Carregando Grafo e Extraindo Comunidades...
Total de 479 macro-comunidades mapeadas na topologia.

2. Carregando Corpus Dinamicamente (Apenas posts válidos para NLP)...
Posts carregados: 1665371
Quantidade de subreddts: 7341
Posts retidos no backbone estrutural: 1109867

3. Carregando Dados do Perspective API e Cruzando...
Posts cruzados com sucesso: 1109867

4. Agregando Estatísticas de Toxicidade por Comunidade...

TOP 10 COMUNIDADES MAIS TÓXICAS DO ECOSSISTEMA
 community         hub_principal  num_subs  num_posts  avg_toxicity  avg_identity
       362                 sissy         2        583      0.630151      0.219407
       351       gonewildstories         4       1192      0.594703      0.184598
       122     gaysexconfessions         2        592      0.570156      0.194160
       383          prostateplay         2        573      0.523683      0.107038
       415            theredpill         2        588      0.466389      0.193996
       249        shitredditsays     